# Assignment 14: Pre-training a Tiny LLM

Fill in every `# TODO` and answer every question in the **Your answer** cells. Run all cells before submitting.

This notebook needs PyTorch (`pip install torch`, or use Google Colab). Training the GPT in Step 5 takes about 8–15 minutes on a laptop CPU.

## Step 1. Setup and data

We train on **TinyStories**: short stories written with the vocabulary of a 3–4-year-old. Simple text lets a tiny model learn real words and grammar.

The cell downloads the file once (22 MB), keeps the first 2 MB, and saves it as `tinystories.txt`.

In [ ]:
import os
import math
import urllib.request
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"
if not os.path.exists("tinystories.txt"):
    full_path, _ = urllib.request.urlretrieve(url)
    with open(full_path, encoding="utf-8") as f:
        raw = f.read()[:2_000_000]
    os.remove(full_path)
    with open("tinystories.txt", "w", encoding="utf-8") as f:
        f.write(raw.replace("<|endoftext|>", "\n"))

with open("tinystories.txt", encoding="utf-8") as f:
    text = f.read()
print("Number of characters:", len(text))
print(text[:400])

## Step 2. The tokenizer

Every character is one token. We use a **fixed** vocabulary, a newline plus the 95 printable ASCII characters, so that the models in Assignments 15 and 16 use the same token IDs. Characters outside the vocabulary are dropped.

### Question 1. Encode and decode

Complete `encode` (text → list of IDs) and `decode` (list of IDs → text).

In [ ]:
chars = ["\n"] + [chr(i) for i in range(32, 127)]
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    # TODO: return the list of IDs of the characters of s that are in the vocabulary

def decode(ids):
    # TODO: return the text for a list of IDs

print("Vocabulary size:", vocab_size)
print(encode("Hi Lily!"), "->", decode(encode("Hi Lily!")))

### Question 2. Training examples

The text labels itself: for every position, the correct answer is the **next** character.

a. Complete `get_batch`: pick `batch_size` random starting positions and return the inputs `x` (chunks of `context_size` characters) and the targets `y` (the same chunks shifted one character to the right).

b. A chunk of `context_size = 128` characters gives how many training examples (context → next character)? Explain.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, batch_size, context_size):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - context_size - 1, (batch_size,))
    # TODO: build x and y with torch.stack
    return x, y

xb, yb = get_batch("train", batch_size=2, context_size=16)
print(repr(decode(xb[0].tolist())), "->", repr(decode(yb[0].tolist())))

**Your answer:** *Write your answer here.*

## Step 3. A baseline: the bigram model

The bigram model predicts the next character from the **current character only**. It is a lookup table with one row of 96 scores per character.

### Question 3. The loss of random guessing

Before training, a model that knows nothing gives every character the same probability $1/96$. What is its cross-entropy loss? Compute it and compare it with the untrained bigram's loss below.

In [ ]:
class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.table(idx)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

bigram = BigramModel(vocab_size)
_, loss = bigram(*get_batch("train", 32, 8))
# TODO: compute the loss of uniform guessing over vocab_size characters
print(f"Untrained bigram loss: {loss.item():.2f}   uniform guessing: {uniform_loss:.2f}")

**Your answer:** *Write your answer here.*

### Question 4. Train the bigram and let it write

Complete the training loop. The four steps are the same as for any neural network: get a batch, compute the loss, backpropagate, update.

In [ ]:
@torch.no_grad()
def estimate_loss(model, batch_size, context_size, eval_batches=50):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        out[split] = sum(model(*get_batch(split, batch_size, context_size))[1].item()
                         for _ in range(eval_batches)) / eval_batches
    model.train()
    return out

def train(model, steps, batch_size, context_size, learning_rate, report_every=500):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []
    for step in range(steps):
        if step % report_every == 0 or step == steps - 1:
            losses = estimate_loss(model, batch_size, context_size)
            history.append((step, losses["train"], losses["val"]))
            print(f"step {step:5d}:  train loss {losses['train']:.3f}   val loss {losses['val']:.3f}")
        # TODO: one training step (batch, loss, zero_grad, backward, step)
    return history

bigram_history = train(bigram, steps=3000, batch_size=32, context_size=8, learning_rate=1e-2)

### Question 5. Sampling with a temperature

To write, the model repeats: predict the scores for the next character, turn them into probabilities, **randomly pick** one, append it.

The **temperature** $T$ divides the scores before the softmax. Complete `generate`, then answer after Question 9: what does a low temperature (0.5) do to the text? A high one (1.5)? Why?

In [ ]:
@torch.no_grad()
def generate(model, max_new_chars, context_size, start="\n", temperature=1.0):
    model.eval()
    idx = torch.tensor([encode(start)], dtype=torch.long)
    for _ in range(max_new_chars):
        logits, _ = model(idx[:, -context_size:])
        # TODO: take the scores of the last position, divide by the temperature, softmax, sample one ID, append it to idx
    model.train()
    return decode(idx[0].tolist())

print(generate(bigram, 300, context_size=8, start="Once upon a time"))

## Step 4. Attention and the tiny GPT

### Question 6. One attention head

Complete the `forward` method of `Head`:

1. Compute the keys, queries and values.
2. Scores = queries × keysᵀ, scaled by $1/\sqrt{\text{head size}}$.
3. Set the scores of **future** positions to $-\infty$ (use `self.mask`).
4. Softmax over the last dimension, then return weights × values.

In [ ]:
class Head(nn.Module):
    def __init__(self, embed_size, head_size, context_size):
        super().__init__()
        self.key = nn.Linear(embed_size, head_size, bias=False)
        self.query = nn.Linear(embed_size, head_size, bias=False)
        self.value = nn.Linear(embed_size, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x):
        B, T, C = x.shape
        # TODO: the four steps above

# A quick check: position 0 can only see itself, so its output must equal its own value vector
head = Head(embed_size=8, head_size=4, context_size=5)
x = torch.randn(1, 5, 8)
print("Position 0 only looks at itself:", torch.allclose(head(x)[0, 0], head.value(x)[0, 0]))

### Question 7. The transformer block

Complete `Block.forward`: attention then feed-forward, each applied to a **LayerNorm** of `x` and **added back** to `x` (a residual connection).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads, context_size):
        super().__init__()
        head_size = embed_size // num_heads
        self.heads = nn.ModuleList([Head(embed_size, head_size, context_size) for _ in range(num_heads)])
        self.proj = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        return self.proj(torch.cat([head(x) for head in self.heads], dim=-1))


class FeedForward(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(embed_size, 4 * embed_size), nn.ReLU(), nn.Linear(4 * embed_size, embed_size))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, embed_size, num_heads, context_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_size)
        self.attention = MultiHeadAttention(embed_size, num_heads, context_size)
        self.ln2 = nn.LayerNorm(embed_size)
        self.feed_forward = FeedForward(embed_size)

    def forward(self, x):
        # TODO: two residual connections
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_layers, context_size):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, embed_size=embed_size, num_heads=num_heads,
                           num_layers=num_layers, context_size=context_size)
        self.context_size = context_size
        self.token_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(context_size, embed_size)
        self.blocks = nn.Sequential(*[Block(embed_size, num_heads, context_size) for _ in range(num_layers)])
        self.ln_final = nn.LayerNorm(embed_size)
        self.output = nn.Linear(embed_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))
        logits = self.output(self.ln_final(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

### Question 8. Count the parameters

The model below has `embed_size = 96`, 4 heads, 4 layers and a context of 128 characters.

a. By hand, compute the number of parameters of the **token embedding**, the **position embedding** and the **output layer**.

b. Use the printout to find what fraction of all parameters are in the 4 transformer blocks.

In [ ]:
context_size, embed_size, num_heads, num_layers = 128, 96, 4, 4
torch.manual_seed(42)
model = TinyGPT(vocab_size, embed_size, num_heads, num_layers, context_size)

for name, part in [("token embedding", model.token_embedding), ("position embedding", model.position_embedding),
                   ("4 transformer blocks", model.blocks), ("final LayerNorm", model.ln_final), ("output layer", model.output)]:
    print(f"{name:<22} {sum(p.numel() for p in part.parameters()):>8,}")
print(f"{'total':<22} {sum(p.numel() for p in model.parameters()):>8,}")

**Your answer:** *Write your answer here.*

## Step 5. Pre-train the tiny GPT

### Question 9. Train and compare

Train the model for 4,000 steps. **This takes about 8–15 minutes on a laptop CPU.**

a. Compare its final validation loss with the bigram's. Convert both to **perplexity** ($e^{\text{loss}}$) and explain what perplexity means here.

b. Generate text at temperatures 0.5, 1.0 and 1.5 and answer the question from Question 5.

In [ ]:
history = train(model, steps=4000, batch_size=32, context_size=context_size, learning_rate=1e-3)

steps = [h[0] for h in history]
plt.plot(steps, [h[1] for h in history], label="training loss")
plt.plot(steps, [h[2] for h in history], label="validation loss")
plt.axhline(bigram_history[-1][2], ls="--", color="gray", label="bigram (validation)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

# TODO: print the perplexity of the bigram and of the tiny GPT (validation)

In [ ]:
for temperature in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    print(f"--- temperature {temperature}")
    print(generate(model, 300, context_size, start="Once upon a time", temperature=temperature), "\n")

**Your answer:** *Write your answer here.*

### Question 10. Does the model really use the context?

The cell below measures the model's average loss **at each position** of a 128-character chunk. Position 0 has no context; position 127 has 127 earlier characters. Describe the curve. What does it tell you about attention?

In [ ]:
@torch.no_grad()
def loss_by_position(model, batches=20):
    model.eval()
    total = torch.zeros(context_size)
    for _ in range(batches):
        x, y = get_batch("val", 64, context_size)
        logits, _ = model(x)
        total += F.cross_entropy(logits.transpose(1, 2), y, reduction="none").mean(dim=0)
    model.train()
    return total / batches

plt.plot(loss_by_position(model))
plt.xlabel("position in the context (number of earlier characters)"); plt.ylabel("validation loss")
plt.title("More context, better predictions"); plt.show()

**Your answer:** *Write your answer here.*

### Question 11. A base model

Give the model a question: `"Q: What color is the sky?\nA:"`. Does it answer? Explain why, using the words **base model** and **next-token prediction**.

In [ ]:
torch.manual_seed(0)
print(generate(model, 200, context_size, start="Q: What color is the sky?\nA:"))

**Your answer:** *Write your answer here.*

## Step 6. Save the model

The next assignment starts from this model. The cell saves it as `base_model.pt`, in the same format that `tiny_gpt.load_model` reads.

In [ ]:
torch.save({"config": model.config, "state_dict": model.state_dict()}, "base_model.pt")
print("saved base_model.pt")